This tutorial can be used to convert the output model files from scPoli to a single .pt file to upload to ArchMap

In [ ]:
!pip install scvi
!pip install scarches

In [ ]:
import scarches as sca
import torch
from scvi.model.base._constants import SAVE_KEYS
import os
import scvi

Below, please specify the directory where your scPoli model files (attr.pkl, model_params.pt, and var_names.csv) are saved. By default the directory is set to the current working directory.

In [ ]:
# load scPoli model
attr_dict, model_state_dict, var_names = sca.models.scPoli._load_params("../benchmark_atlas/models/reference_retraining1", map_location="cpu")
model = sca.models.scPoli.load("../benchmark_atlas/models/reference_retraining1", map_location="cpu")

In [ ]:
model.adata

In [114]:
init_params_ = model._get_init_params_from_dict(attr_dict)
setup_keys = ["condition_keys","cell_type_keys"]
setup_args = {k: v for k, v in init_params_.items() if k in setup_keys}
keys_to_drop=["condition_keys","cell_type_keys","share_metadata","obs_metadata","prototypes_labeled","cov","conditions","cell_types","conditions_combined","labeled_indices"]
init_params = {k: v for k, v in init_params_.items() if k not in keys_to_drop}
attr_dict["init_params_"]=init_params

In [115]:
attr_dict["registry_"]={}
attr_dict["registry_"]["scvi_version"]="n/a"
attr_dict["registry_"]["model_name"]="SCPOLI"
attr_dict["registry_"]["setup_args"]=setup_args
attr_dict["registry_"]["field_registries"]={}
attr_dict["registry_"]["field_registries"]["summary_stats"]={}
attr_dict["registry_"]["field_registries"]["data_registry"]={}

In [116]:
# add summary stats
attr_dict["registry_"]["field_registries"]={'X':{}, 'batch':{}, 'labels':{}, 'size_factor':{}, 'extra_categorical_covs':{}, 'extra_continuous_covs':{}}
attr_dict["registry_"]["field_registries"]["X"]["summary_stats"]={'n_vars': model.adata.n_vars, 'n_cells': model.adata.n_obs}
attr_dict["registry_"]["field_registries"]["batch"]["summary_stats"]={'n_batch': len(attr_dict["conditions_combined_"])}
attr_dict["registry_"]["field_registries"]["labels"]["summary_stats"]={'n_labels': len(attr_dict["cell_types_"].keys())}
attr_dict["registry_"]["field_registries"]["size_factor"]["summary_stats"]={}
attr_dict["registry_"]["field_registries"]["extra_categorical_covs"]["summary_stats"]={}
attr_dict["registry_"]["field_registries"]["extra_continuous_covs"]["summary_stats"]={}

In [117]:
# add data registry
for registry_key, field_registry in attr_dict["registry_"]["field_registries"].items():
    field_registry["data_registry"] = {}

In [ ]:
from rich.console import Console
console = Console(no_color=True)
console.print("Summary Stat Key | Value")

In [137]:
from scvi.data import AnnDataManager
model_summary_stats = dict(AnnDataManager._get_summary_stats_from_registry(attr_dict["registry_"]))

In [ ]:
model_summary_stats

In [139]:
model_summary_stats=AnnDataManager._view_summary_stats(
                model_summary_stats, as_markdown=True
            ),

In [ ]:
model_summary_stats

Below you can optionally set a specific file output path and prefix. By default the output model will be saved in the current working directory under the name "model.pt"

In [11]:
# specify output path and output file prefix (default to current working directory with no prefix)
dir_path = "../benchmark_atlas/models/reference_retraining1"
file_name_prefix = ""
save_kwargs={}

model_save_path = os.path.join(dir_path, f"{file_name_prefix}{SAVE_KEYS.MODEL_FNAME}")

# only save the public attributes with _ at the very end
user_attributes = {k: v for k, v in attr_dict.items() if k[-1] == "_"}


torch.save(

{

SAVE_KEYS.MODEL_STATE_DICT_KEY: model_state_dict,

SAVE_KEYS.VAR_NAMES_KEY: var_names,

SAVE_KEYS.ATTR_DICT_KEY: user_attributes,

},

model_save_path,

**save_kwargs,

)

In [ ]:
!pip install huggingface_hub

In [ ]:
from scvi.hub import HubMetadata, HubModel, HubModelCardHelper
import anndata
model_path = "classifiers/data/model_hnoca_scpoli"

hm = HubMetadata.from_dir(model_path, anndata_version=anndata.__version__, map_location='cpu')


In [ ]:
hmch = HubModelCardHelper.from_dir(
    model_path,
    license_info="cc-by-4.0",
    anndata_version=anndata.__version__,
    data_modalities=["rna"],
    data_is_annotated=True,
    description="HNOCA atlas",
    model_parent_module="scarches.model",
    data_is_minified=True,
    references="He, Z., Dony, L., Fleck, J.S. et al. An integrated transcriptomic cell atlas of human neural organoids. Nature 635, 690–698 (2024). https://doi.org/10.1038/s41586-024-08172-8",
)

In [141]:
hmch.model_card.save(
    "my_model_card.md"
)  # then change the markdown file on disk...

In [ ]:
print(hmch.model_card.content)

Create HubModel and upload it

In [ ]:
hmo = HubModel(model_path, metadata=hm, model_card=hmch)
hmo

In [ ]:
hmo.push_to_huggingface_hub(
    repo_name="scvi-tools/human-neural-organoid-cell-atlas-scpoli", repo_token="", repo_create=True
)